# Gate features run: all official photos, 6 queries, per-photo gate features
Same pipeline as `tests/Heading_Queries` (edge ZNCC + DINO layer + LoFTR-MINIMA, 3 headings x 2 scales), but run on every photo of the pool, saving the numbers a decision gate can use (`train/checks/gate_features.py`). Resumable: rows are appended to a CSV every photo.

In [ ]:
import os, sys, math, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent                    # walk up to the repo folder
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

# ------------------------------------------------------------------ settings: change here, run again
DATA = "official"           # "official" (d1 photos; the map comes from src/header/map_source.py) or "nudt" (Changsha test photos)
MAP = "new"                 # base map of the official photos: "new" (both scenes), "new_931", "new_921" or "old"
MAP_MARGIN_M = None         # metres of map kept around the photos (None = 6000 from map_source.py; the search disc is 5500 m)
POOL_PHOTOS = 500           # photos loaded once; the seeds pick from them
SEEDS = [11]        # three random seeds, side by side: each picks its own photos
N_PER_SEED = 150            # photos per seed (a multiple of 3: the same number from each height group). 150 x 3 seeds covers about 330 of the ~500 photos, so the seeds overlap and the 'all' column is the one to trust for the size of the gain
FOUND_SIGMA = 4.0           # a place counts as found when it is within this many sigma of the truth

# the queries: every photo is drawn once per (heading, scale factor)
HEADINGS = [-5.0, 0.0, 5.0]   # degrees added to the yaw (Patch v0.5.6: the needed corrections are within +-10)
SCALE_FACTORS = [1.0, 1.05]       # the projection scale is multiplied by each of these; [1.0, 1.05] doubles the queries (level 7 photos preferred 1.05)

# per query: the same pipeline as Single_Image_Run
COARSE = "edge_zncc"        # "edge_zncc" (one map, cheap: the patch recommends it for the hypotheses) or "fused" (six maps, about 9 times that cost)
USE_SEMANTIC_LAYER = True   # DINO re-orders the K coarse places of each query
K_PER_QUERY = 50            # K: places the coarse search returns for each query
M_PER_QUERY = 10            # M: places of each query that go to LoFTR (5 gave the same score as 25 on the development photos)
DINO_WEIGHT = None          # None = 100 (DINO decides the order inside the K)
GAMMA = 1.0                 # the answer is the candidate with the most  inliers + GAMMA * (DINO z-score) among ALL queries
MATCHER = "loftr_minima"    # "lightglue", "loftr_minima" or "roma_minima"
MATCHER_CONFIDENCE = 0.22   # LoFTR only: matches below this confidence are dropped (0.2 = kornia's default; lower lets weak, possibly false matches through)
LOFTR_COARSE_THRESHOLD = 0.21  # LoFTR only: its own cut on the coarse match probability
RANSAC_PIXELS = None        # None = 3.0
MIN_INLIERS = 8             # answer bar

# the variants that are scored from the same run: name -> (headings, scale factors); a variant needs all its queries to exist
VARIANTS = {
    "plain (heading 0)": ([0.0], [1.0]),
    "3 headings (-5, 0, +5)": ([-5.0, 0.0, 5.0], [1.0]),
    "5 headings (step 5)": ([-10.0, -5.0, 0.0, 5.0, 10.0], [1.0]),
    "all headings": (HEADINGS, [1.0]),
    "all headings x all scales": (HEADINGS, SCALE_FACTORS),
}
# the same candidates are also scored with a cost for the heading: each candidate pays  penalty * |heading|  inliers (inliers per degree) when the best one is chosen.
# 0 = the best candidate wins whatever its heading (the variants above). These are scored on the "all headings" queries from the same run, nothing is run twice.
HEADING_PENALTIES = []     # 0.5 means a candidate 10 degrees off needs 5 more inliers than one at 0 degrees
plt.rcParams["figure.dpi"] = 90

assert len(SEEDS) >= 1 and N_PER_SEED >= 3, "need at least one seed and 3 photos per seed"
assert 0.0 in HEADINGS and 1.0 in SCALE_FACTORS, "the plain query (heading 0, scale factor 1) is the baseline and must be in the lists"

In [ ]:
from heading_query_pipeline import HeadingQueryPipeline        # first import: it saves the development-set settings before NUDT changes them
from header import settings
from header.sampling import pick_sample_ids
from header.scoring import final_score
from coarse_search.heading_queries import query_list
from train.checks.heading_results import variant_ids, query_rows, variant_rows, summarise

pipeline = HeadingQueryPipeline(DATA, COARSE, semantic_layer=USE_SEMANTIC_LAYER, semantic_decision=True, gamma=GAMMA,
                                k_peaks=K_PER_QUERY, m_fine=M_PER_QUERY, matcher=MATCHER, min_inliers=MIN_INLIERS, ransac_pixels=RANSAC_PIXELS,
                                matcher_confidence=MATCHER_CONFIDENCE, satellite=MAP, map_margin_m=MAP_MARGIN_M,
                                loftr_coarse_threshold=LOFTR_COARSE_THRESHOLD, dino_weight=DINO_WEIGHT,
                                headings=HEADINGS, scale_factors=SCALE_FACTORS)
start = time.time()
pipeline.setup(POOL_PHOTOS, 7)
print("setup done in %.1f s" % (time.time() - start))
for line in pipeline.describe():
    print(line)

test_set = pipeline.test_set
camera = pipeline.camera

In [ ]:
import torch, pandas as pd
from train.checks.gate_features import gate_row
OUT = ROOT / ("tests/Test_Iteration_3/results/fresh_run/gate_rows_conf%.2f.csv" % MATCHER_CONFIDENCE)
done = set(pd.read_csv(OUT).photo) if OUT.exists() else set()
n_all = len(test_set.items); start = time.time(); failures = []
for index in range(n_all):
    if index in done:
        continue
    item = test_set.items[index]
    try:
        found = None
        for attempt in range(3):
            try:
                found = pipeline.run_queries(item, test_set.priors[index]); break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); time.sleep(5)
        if found is None:
            raise MemoryError("GPU out of memory three times")
        answer = pipeline.answer_for(found["queries"], [q["id"] for q in found["queries"]])
        row = gate_row(found, answer, test_set.truths[index], test_set.sigmas[index], item["rel_height"])
        row["photo"] = index
        pd.DataFrame([row]).to_csv(OUT, mode="a", header=not OUT.exists(), index=False)
    except Exception as error:
        failures.append((index, repr(error)[:150])); print("photo", index, "failed", repr(error)[:150])
    if index % 10 == 0:
        print("photo %d/%d | %.0f s" % (index, n_all, time.time() - start), flush=True)
print("done", n_all, "photos;", len(failures), "failed")